## Splitting and Embedding Text Using LangChain

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

True

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


with open('/home/aswanth.cp@knackforge.com/Public/Tutorial/udemy/langchain/churchill_speech.txt') as f:
    churchill_speech = f.read()
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 100,
        chunk_overlap = 20,
        length_function = len
    )

In [ ]:
chunks = text_splitter.split_text(churchill_speech)
print(len(chunks))

300
Winston Churchill Speech - We Shall Fight on the Beaches
We Shall Fight on the Beaches
June 4, 1940


## Embedding Cost

In [11]:
def print_embedding_cost(docs):
    import tiktoken
    
    COST_PER_1K = 0.00002
    
    enc = tiktoken.encoding_for_model("text-embedding-3-small")
    
    total_tokens = sum(len(enc.encode(doc)) for doc in docs)
    
    cost = (total_tokens / 1000) * COST_PER_1K
    
    print(f"Total tokens: {total_tokens}")
    print(f"Estimated embedding cost in USD: ${cost:.6f}")

print_embedding_cost(chunks)

Total tokens: 4820
Estimated embedding cost in USD: $0.000096


In [30]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1125.56it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [17]:
vector = embeddings.embed_query(chunks[0])
vector[:5]

[0.05969041958451271,
 0.05481327325105667,
 -0.01299942098557949,
 0.013887403532862663,
 0.03500545024871826]

In [20]:
import os 
import pinecone



pc = pinecone.Pinecone()

In [33]:
# deleting all indexes
indexes = pc.list_indexes().names()
for i in indexes:
    print('Deleting all indexes ... ', end='')
    pc.delete_index(i)
    print('Done')

Deleting all indexes ... Done


In [34]:
from pinecone import ServerlessSpec

index_name = 'churchill-speech'

if index_name not in pc.list_indexes().names():
    print(f'Creating index: {index_name}')
    pc.create_index(
        name=index_name,
        dimension=384,
        metric='cosine',
        spec=ServerlessSpec(
            cloud='aws',
            region='us-east-1',
        )

    )
    print(f'Index {index_name} created successfully.')
else:
    print(f'Index {index_name} already exists.')



Creating index: churchill-speech
Index churchill-speech created successfully.


In [35]:
from langchain_pinecone import PineconeVectorStore
from langchain_core.documents import Document

docs = [Document(page_content=text) for text in chunks]

vectorstore_from_docs = PineconeVectorStore.from_documents(
        docs,
        index_name=index_name,
        embedding=embeddings
    )

In [37]:
index = pc.Index(index_name)

index.describe_index_stats()

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '135',
                                    'content-type': 'application/json',
                                    'date': 'Mon, 02 Mar 2026 09:24:23 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '37',
                                    'x-pinecone-request-latency-ms': '37',
                                    'x-pinecone-response-duration-ms': '39'}},
 'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 300}},
 'total_vector_count': 300,
 'vector_type': 'dense'}

In [41]:
query = 'Where should we fight?'
result = vectorstore_from_docs.similarity_search(query)
print(result)

[Document(id='7b5df523-7d7e-494f-8e37-9a1b6d51c4b0', metadata={}, page_content='shall fight on the beaches, we shall fight on the landing grounds, we shall fight in the fields and'), Document(id='735f7510-ce7d-42aa-af53-e951c5961aaa', metadata={}, page_content='end, we shall fight in France, we shall fight on the seas and oceans, we shall fight with growing'), Document(id='bc8bffea-537b-4727-9f2d-03f89f67801c', metadata={}, page_content='front, now on that, fighting'), Document(id='6390e0a2-d79e-43c2-8f3b-787cb4bbdfef', metadata={}, page_content='with a defensive war. We have our duty to our Ally. We have to reconstitute and build up the')]


In [42]:
for r in result:
    print(r.page_content)
    print('-' * 50)

shall fight on the beaches, we shall fight on the landing grounds, we shall fight in the fields and
--------------------------------------------------
end, we shall fight in France, we shall fight on the seas and oceans, we shall fight with growing
--------------------------------------------------
front, now on that, fighting
--------------------------------------------------
with a defensive war. We have our duty to our Ally. We have to reconstitute and build up the
--------------------------------------------------


In [ ]:

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

retriever = vectorstore_from_docs.as_retriever(search_type='similarity', search_kwargs={'k': 3})

# --- 4. Define LLM & Prompt ---

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv('GROQ_API_KEY'),
)


template = """
Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)


rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# --- 6. Execution ---
response = rag_chain.invoke("who is the speech about?")
print(response)


The speech is by Winston Churchill, as indicated in the page_content of the first document: "Winston Churchill Speech - We Shall Fight on the Beaches".


In [48]:
query = 'Answer only from the provided input. Where should we fight?'
answer = rag_chain.invoke(query)
print(answer)

According to the provided context, we should fight on the "front".


In [49]:
query = 'Who was the king of Belgium at that time?'
answer = rag_chain.invoke(query)
print(answer)

The text does not explicitly mention the name of the King of Belgium at that time. It only refers to "the Belgian King" or "the King of the Belgians", but does not provide a specific name.


In [50]:
query = 'What about the French Armies??'
answer = rag_chain.invoke(query)
print(answer)

The context mentions two French Armies: 

1. The French First Army: It seemed that the whole of the French First Army was going to "be reembarked".
2. Another French Army (without a specified number): It was mentioned that the British and French Armies fought, and some area was held by the French troops.


In [51]:
from langchain_core.runnables import RunnableParallel

rag_chain_with_sources = RunnableParallel(
    {"context": retriever, "question": RunnablePassthrough()}
).assign(
    answer=(
        RunnablePassthrough.assign(
            context=lambda x: [doc.page_content for doc in x["context"]]
        )
        | prompt
        | llm
        | StrOutputParser()
    )
)

result = rag_chain_with_sources.invoke("Who was the king of Belgium?")

print(f"Answer: {result['answer']}")
print("\n--- Sources Used ---")
for doc in result['context']:
    print(f"- {doc.page_content[:150]}...")



Answer: King Leopold

--- Sources Used ---
- French Armies who had entered Belgium at the appeal of the Belgian King; but this strategic fact...
- fall upon us. The King of the Belgians had called upon us to come to his aid. Had not this Ruler...
- the last moment, when Belgium was already invaded, King Leopold called upon us to come to his aid,...
